# Installation

In [ ]:
%%capture
# Normally using pip install unsloth is enough

# Temporarily as of Jan 31st 2025, Colab has some issues with Pytorch
# Using pip install unsloth will take 3 minutes, whilst the below takes <1 minute:
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post2 peft trl triton
!pip install --no-deps cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_PATH = './'

In [ ]:
PROJECT_NAME = "mesos"

# Data prep

In [ ]:
# deterministic sampling 80/20 train/test
import pandas as pd

df_all_tickets = pd.read_csv('{}datasets/{}.csv'.format(BASE_PATH, PROJECT_NAME))
train_split_point = int(len(df_all_tickets) * 0.8)

df_tickets = df_all_tickets.iloc[:train_split_point,]
df_tickets_test = df_all_tickets.iloc[train_split_point:,]

df_tickets

,Unnamed: 0,Summary,Custom field (Story Points),Assignee_count,Reporter_count,Creator_count
0,0,Argument forwaring in CMake build result in gl...,1.0,0.000000,0.014286,0.014286
1,1,MasterQuotaTest.ValidateLimitAgainstConsumed i...,3.0,0.000000,1.000000,1.000000
2,2,Re-evaluate libevent SSL socket EOF semantics ...,3.0,0.000000,1.000000,1.000000
3,3,Mesos may report completed task as running in ...,5.0,0.000000,0.200000,0.200000
4,4,Flapping tasks with large sandboxes can fill a...,5.0,0.000000,1.000000,1.000000
...,...,...,...,...,...,...
326,326,Consider including public protobuf definitions...,1.0,0.000000,0.185714,0.185714
327,327,Restructure Web UI to make updates simpler to do.,3.0,0.071429,0.000000,0.000000
328,328,Add Docker inspect timeout when collecting con...,3.0,0.000000,1.000000,1.000000
329,329,UCR doesn't isolate uts namespace w/ host netw...,3.0,0.214286,0.000000,0.000000


In [ ]:
df_tickets = df_tickets.rename(columns={'Custom field (Story Points)': 'storypoint'})
df_tickets_test = df_tickets_test.rename(columns={'Custom field (Story Points)': 'storypoint'})

# fine tune

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# More models at https://huggingface.co/unsloth
base_model = "unsloth/Llama-3.1-8B-Instruct"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [ ]:
rank = 8

model_peft = FastLanguageModel.get_peft_model(
    model,
    # LoRA rank. Higher rank = more trainable parameters but better model quality
    r = rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    # List of model layers to apply LoRA to (attention and feed-forward layers)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    # Scaling factor for LoRA updates. Usually set equal to rank (r)
    lora_alpha = rank,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 42,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.3.19 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
import json

system_prompt = """You are an expert software development effort estimator. Given a software development issue summary, predict the effort in story points."""
user_prompt = """### Summary:
{}"""
assistant_prompt = """My estimated story point is: {}"""

def generate_convo(row):
  # {"role": "system", "content": "You are an assistant"}
  # {"role": "user", "content": "What is 2+2?"}
  # {"role": "assistant", "content": "It's 4."}

  convo_template = {
    "conversations": [
      {
          "role": "system",
          "content": system_prompt
      },
      {
          "role": "user",
          "content": user_prompt.format(
                          row['Summary'],
                      )
      },
      {
          "role": "assistant",
          "content": assistant_prompt.format(
                          int(row['storypoint'])
                      )
      }
    ]
  }

  return convo_template

print(json.dumps(generate_convo(df_tickets.iloc[0]), indent=2))

{
  "conversations": [
    {
      "role": "system",
      "content": "You are an expert software development effort estimator. Given a software development issue summary, predict the effort in story points."
    },
    {
      "role": "user",
      "content": "### Summary:\nArgument forwaring in CMake build result in glog 0.4.0 build as shared library"
    },
    {
      "role": "assistant",
      "content": "My estimated story point is: 1"
    }
  ]
}


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(tasks):
    convos = [generate_convo(task) for _, task in tasks.iterrows()]
    texts = [tokenizer.apply_chat_template(convo['conversations'], tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

In [ ]:
from datasets import Dataset

story_point_texts = formatting_prompts_func(df_tickets)
combined_texts = { "text" : story_point_texts["text"] }
combined_dataset = Dataset.from_dict(combined_texts)
print(combined_dataset[0])

{'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nYou are an expert software development effort estimator. Given a software development issue summary, predict the effort in story points.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n### Summary:\nArgument forwaring in CMake build result in glog 0.4.0 build as shared library<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nMy estimated story point is: 1<|eot_id|>'}


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

train_dataset = combined_dataset
# max_steps = 50
epoch = 10

trainer = SFTTrainer(
    model = model_peft,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2, # Uses 2 processes for data loading.
    packing = True, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
	      # increase = more VRAM consumed
        per_device_train_batch_size = 4, # typical = 8
        # gradients calculated from each batch are accumulated
        # over 4 iterations before an optimizer step is performed.
        gradient_accumulation_steps = 4,
        # gradually increase the learning rate at the beginning
        # of training, helps to stabilize training and prevent
        # the model from diverging early on.
        warmup_steps = 10,
        num_train_epochs = epoch, # Set this for 1 full training run.
        # max_steps = max_steps,
        learning_rate = 2e-4, # usually closer to 2e-4 through 2e-6.
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "wandb", # Use this for WandB etc
        # -----------------
        # evaluation_strategy="steps", # Evaluate the model every logging step
        logging_dir="./logs",        # Directory for storing logs
        # save_strategy="steps",       # Save the model checkpoint every logging step
        save_strategy="epoch",       # Save the model checkpoint every logging step
        # eval_steps=10,               # Evaluate and save checkpoints every 10 steps
        # do_eval=True                 # Perform evaluation at the end of training
        # save_steps=20               # Save every 10 checkpoints
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/331 [00:00<?, ? examples/s]

Unsloth: Hugging Face's packing is currently buggy - we're disabling it for now!


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
allocated_memory = round(torch.cuda.memory_allocated()/ 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. VRAM {allocated_memory}/{max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. VRAM 5.662/39.557 GB.
5.67 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

6.11 minutes used for training.
Peak reserved memory = 6.344 GB.
Peak reserved memory for training = 0.674 GB.
Peak reserved memory % of max memory = 16.038 %.
Peak reserved memory for training % of max memory = 1.704 %.


In [ ]:
export_model_name = f"{PROJECT_NAME}_{base_model.split('/')[-1]}_e{epoch}_msl{max_seq_length}_r{rank}"

model_peft.save_pretrained(BASE_PATH + "models/" + export_model_name)
tokenizer.save_pretrained(BASE_PATH + "models/" + export_model_name)